In [7]:
from bsk_rl import data, obs, act, scene, sats, comm
from bsk_rl.sim import dyn, fsw

class ImagingSatellite(sats.ImagingSatellite):
    observation_spec=[
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=10,
        )
    ]
    action_spec=[act.Image(n_ahead_image=10)]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel
    

In [8]:
from bsk_rl.utils.orbital import walker_delta_args

sat_args = dict(
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
    dataStorageCapacity=1e12,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    omega_max=0.087,
    servo_Ki=5.0,
    servo_P=150 / 5,
)
sat_arg_randomizer = walker_delta_args(altitude=800.0, inc=60.0, n_planes=1)

Gym API

In [9]:
from bsk_rl import GeneralSatelliteTasking

env = GeneralSatelliteTasking(
    satellites=[
        ImagingSatellite("EO-1", sat_args),
        ImagingSatellite("EO-2", sat_args),
        ImagingSatellite("EO-3", sat_args),
    ],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=comm.LOSCommunication(),
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)

env.reset()

env.observation_space

2025-12-08 14:39:08,294                                WARNING    Creating logger for new env on PID=30028. Old environments in process may now log times incorrectly.
2025-12-08 14:39:08,719 gym                            INFO       Resetting environment with seed=1300272708
2025-12-08 14:39:08,721 scene.targets                  INFO       Generating 1000 targets
2025-12-08 14:39:08,895 sats.satellite.EO-1            INFO       <0.00> EO-1: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 14:39:08,917 sats.satellite.EO-2            INFO       <0.00> EO-2: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 14:39:08,939 sats.satellite.EO-3            INFO       <0.00> EO-3: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 14:39:08,966 gym                            INFO       <0.00> Environment reset


Tuple(Box(-1e+16, 1e+16, (20,), float64), Box(-1e+16, 1e+16, (20,), float64), Box(-1e+16, 1e+16, (20,), float64))

In [12]:
env.action_space


Tuple(Discrete(10), Discrete(10), Discrete(10))

In [13]:
observation, reward, terminated, truncated, info = env.step([7,8,9])

2025-12-08 14:41:27,442 gym                            INFO       <0.00> === STARTING STEP ===
2025-12-08 14:41:27,442 sats.satellite.EO-1            INFO       <0.00> EO-1: target index 7 tasked
2025-12-08 14:41:27,443 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-375) tasked for imaging
2025-12-08 14:41:27,443 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-375) window enabled: 248.6 to 453.1
2025-12-08 14:41:27,444 sats.satellite.EO-1            INFO       <0.00> EO-1: setting timed terminal event at 453.1
2025-12-08 14:41:27,445 sats.satellite.EO-2            INFO       <0.00> EO-2: target index 8 tasked
2025-12-08 14:41:27,446 sats.satellite.EO-2            INFO       <0.00> EO-2: Target(tgt-210) tasked for imaging
2025-12-08 14:41:27,447 sats.satellite.EO-2            INFO       <0.00> EO-2: Target(tgt-210) window enabled: 410.9 to 581.6
2025-12-08 14:41:27,447 sats.satellite.EO-2            INFO       <0.00> EO-2: setting timed terminal even

In [14]:
observation

(array([ 0.84642446,  0.00446111,  0.10829544, -0.0197192 ,  0.32927575,
         0.01729217,  0.9555779 ,  0.02602833,  0.83044131,  0.07690408,
         0.90952978,  0.09125535,  0.45005708,  0.11241823,  0.22398983,
         0.13821119,  0.18621412,  0.13727625,  0.14816885,  0.20757037]),
 array([ 0.05020028, -0.0044881 ,  0.09420785, -0.02631579,  0.58328142,
        -0.01712015,  0.64517296, -0.01784139,  0.13450343, -0.00752126,
         0.07417278,  0.00214164,  0.58657049,  0.03467345,  0.50646344,
         0.04576515,  0.38686966,  0.06532432,  0.66040506,  0.06082158]),
 array([ 0.27163509, -0.02631579,  0.99299071, -0.00769405,  0.32516576,
        -0.02237468,  0.17617117, -0.02146956,  0.95745677, -0.01471053,
         0.90645087, -0.00943217,  0.82492698, -0.00264029,  0.1548313 ,
         0.01728155,  0.12769496,  0.04060358,  0.38429681,  0.03433514]))

In [15]:
info

{'EO-1': {'requires_retasking': False},
 'EO-2': {'requires_retasking': False},
 'EO-3': {'requires_retasking': True},
 'd_ts': 150.0}

In [16]:
actions = [0 if info[sat.name]["requires_retasking"] else None for sat in env.unwrapped.satellites]
actions

[None, None, 0]

In [17]:
observation, reward, terminated, truncated, info = env.step(actions)

2025-12-08 15:01:44,833 gym                            INFO       <150.00> === STARTING STEP ===
2025-12-08 15:01:44,834 sats.satellite.EO-3            INFO       <150.00> EO-3: target index 0 tasked
2025-12-08 15:01:44,836 sats.satellite.EO-3            INFO       <150.00> EO-3: Target(tgt-777) tasked for imaging
2025-12-08 15:01:44,836 sats.satellite.EO-3            INFO       <150.00> EO-3: Target(tgt-777) window enabled: 0.0 to 164.6
2025-12-08 15:01:44,837 sats.satellite.EO-3            INFO       <150.00> EO-3: setting timed terminal event at 164.6
2025-12-08 15:01:44,856 sats.satellite.EO-3            INFO       <165.00> EO-3: timed termination at 164.6 for Target(tgt-777) window
2025-12-08 15:01:44,857 data.base                      INFO       <165.00> Total reward: {}
2025-12-08 15:01:44,858 sats.satellite.EO-3            INFO       <165.00> EO-3: Satellite EO-3 requires retasking
2025-12-08 15:01:44,860 gym                            INFO       <165.00> Step reward: 0.0


In [18]:
from Basilisk.architecture import messaging

def isnt_alive(log_failure=False):
    """Mock satellite 0 dying."""
    self = env.unwrapped.satellites[0]
    death_message = messaging.PowerStorageStatusMsgPayload()
    death_message.storageLevel = 0.0
    self.dynamics.powerMonitor.batPowerOutMsg.write(death_message)
    return self.dynamics.is_alive(log_failure=log_failure) and self.fsw.is_alive(
        log_failure=log_failure
    )

env.unwrapped.satellites[0].is_alive = isnt_alive
observation, reward, terminated, truncated, info = env.step([6, 7, 9])

2025-12-08 15:04:18,332 gym                            INFO       <165.00> === STARTING STEP ===
2025-12-08 15:04:18,333 sats.satellite.EO-1            INFO       <165.00> EO-1: target index 6 tasked
2025-12-08 15:04:18,333 sats.satellite.EO-1            INFO       <165.00> EO-1: Target(tgt-813) tasked for imaging
2025-12-08 15:04:18,334 sats.satellite.EO-1            INFO       <165.00> EO-1: Target(tgt-813) window enabled: 790.8 to 986.0
2025-12-08 15:04:18,334 sats.satellite.EO-1            INFO       <165.00> EO-1: setting timed terminal event at 986.0
2025-12-08 15:04:18,336 sats.satellite.EO-2            INFO       <165.00> EO-2: target index 7 tasked
2025-12-08 15:04:18,336 sats.satellite.EO-2            INFO       <165.00> EO-2: Target(tgt-210) window enabled: 410.9 to 581.6
2025-12-08 15:04:18,337 sats.satellite.EO-2            INFO       <165.00> EO-2: setting timed terminal event at 581.6
2025-12-08 15:04:18,337 sats.satellite.EO-3            INFO       <165.00> EO-3: target

PettingZOO

In [21]:
from bsk_rl import ConstellationTasking

env = ConstellationTasking(
    satellites=[
        ImagingSatellite(f"EO-{i+1}", sat_args) for i in range(3)
    ],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=comm.LOSCommunication(),
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)

env.reset()
env.observation_spaces

2025-12-08 15:19:31,933                                WARNING    Creating logger for new env on PID=30028. Old environments in process may now log times incorrectly.
2025-12-08 15:19:32,577 gym                            INFO       Resetting environment with seed=2796455964
2025-12-08 15:19:32,579 scene.targets                  INFO       Generating 1000 targets
2025-12-08 15:19:32,760 sats.satellite.EO-1            INFO       <0.00> EO-1: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 15:19:32,785 sats.satellite.EO-2            INFO       <0.00> EO-2: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 15:19:32,808 sats.satellite.EO-2            INFO       <0.00> EO-2: Finding opportunity windows from 600.00 to 1200.00 seconds
2025-12-08 15:19:32,832 sats.satellite.EO-3            INFO       <0.00> EO-3: Finding opportunity windows from 0.00 to 600.00 seconds
2025-12-08 15:19:32,856 gym                            INFO       <0.00> Environment reset


{'EO-1': Box(-1e+16, 1e+16, (20,), float64),
 'EO-2': Box(-1e+16, 1e+16, (20,), float64),
 'EO-3': Box(-1e+16, 1e+16, (20,), float64)}

In [22]:
observation, reward, terminated, truncated, info = env.step(
    {
        env.agents[0]: 7,
        env.agents[1]: 9,
        env.agents[2]: 8,
    }
)

2025-12-08 15:21:40,737 gym                            INFO       <0.00> === STARTING STEP ===
2025-12-08 15:21:40,737 sats.satellite.EO-1            INFO       <0.00> EO-1: target index 7 tasked
2025-12-08 15:21:40,738 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-697) tasked for imaging
2025-12-08 15:21:40,739 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-697) window enabled: 77.2 to 276.5
2025-12-08 15:21:40,739 sats.satellite.EO-1            INFO       <0.00> EO-1: setting timed terminal event at 276.5
2025-12-08 15:21:40,740 sats.satellite.EO-2            INFO       <0.00> EO-2: target index 9 tasked
2025-12-08 15:21:40,741 sats.satellite.EO-2            INFO       <0.00> EO-2: Target(tgt-948) tasked for imaging
2025-12-08 15:21:40,741 sats.satellite.EO-2            INFO       <0.00> EO-2: Target(tgt-948) window enabled: 662.7 to 871.0
2025-12-08 15:21:40,742 sats.satellite.EO-2            INFO       <0.00> EO-2: setting timed terminal event

In [23]:
observation

{'EO-1': array([ 0.9263519 , -0.01403509,  0.00390898, -0.01403509,  0.77556157,
         0.0057284 ,  0.98125027, -0.00440297,  0.13187196,  0.00212192,
         0.52828608,  0.03080046,  0.37522152,  0.0092897 ,  0.54423581,
         0.02318537,  0.70765404,  0.04893848,  0.60449051,  0.05300896]),
 'EO-2': array([ 2.37183780e-01, -1.40350877e-02,  2.90280729e-01, -1.40350877e-02,
         4.06665227e-01, -1.26730700e-02,  6.96089483e-01, -4.68715663e-05,
         5.16042758e-01,  1.24548919e-02,  2.23923087e-01,  3.36446417e-02,
         1.77119997e-01,  4.42078299e-02,  5.26446720e-01,  7.40838644e-02,
         3.99196174e-02,  1.13020452e-01,  6.14174360e-02,  1.02222169e-01]),
 'EO-3': array([ 0.312303  , -0.01403509,  0.58168214, -0.00993629,  0.1255191 ,
        -0.00262157,  0.94592871,  0.00337549,  0.83907957,  0.03750168,
         0.08001654,  0.04700822,  0.28097509,  0.04850005,  0.16964701,
         0.06821733,  0.99283663,  0.08757778,  0.39521865,  0.09531188])}

In [24]:
info

{'EO-1': {'requires_retasking': True, 'd_ts': 80.0},
 'EO-2': {'requires_retasking': False, 'd_ts': 80.0},
 'EO-3': {'requires_retasking': False, 'd_ts': 80.0},
 '__common__': {'d_ts': 80.0}}

In [25]:
# Immediately kill satellite 0
env.unwrapped.satellites[0].is_alive = isnt_alive
env.agents

['EO-1', 'EO-2', 'EO-3']

In [26]:
observation, reward, terminated, truncated, info = env.step({
        env.agents[0]: 7,
        env.agents[1]: 9,
    }
)

2025-12-08 15:24:32,742 gym                            INFO       <80.00> === STARTING STEP ===
2025-12-08 15:24:32,742 sats.satellite.EO-1            INFO       <80.00> EO-1: target index 7 tasked
2025-12-08 15:24:32,743 sats.satellite.EO-1            INFO       <80.00> EO-1: Target(tgt-607) tasked for imaging
2025-12-08 15:24:32,743 sats.satellite.EO-1            INFO       <80.00> EO-1: Target(tgt-607) window enabled: 212.2 to 396.9
2025-12-08 15:24:32,744 sats.satellite.EO-1            INFO       <80.00> EO-1: setting timed terminal event at 396.9
2025-12-08 15:24:32,744 sats.satellite.EO-2            INFO       <80.00> EO-2: target index 9 tasked
2025-12-08 15:24:32,744 sats.satellite.EO-2            INFO       <80.00> EO-2: Target(tgt-948) window enabled: 662.7 to 871.0
2025-12-08 15:24:32,744 sats.satellite.EO-2            INFO       <80.00> EO-2: setting timed terminal event at 871.0
2025-12-08 15:24:32,780 sats.satellite.EO-1            INFO       <215.00> EO-1: imaged Target(